# EDA 05 — Qualite de l'Air (Airparif ATMO)
**Source** : Airparif | **Bronze** : `data/bronze/air_quality/`

In [ ]:

import os, glob, warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
warnings.filterwarnings("ignore")

PALETTE = {
    "primary":   "#1565C0",
    "secondary": "#E53935",
    "accent":    "#2E7D32",
    "neutral":   "#546E7A",
}

plt.rcParams.update({
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.framealpha": 0.9,
    "legend.fontsize": 10,
})

BRONZE_ROOT = os.path.abspath("../../data/bronze")
NB_DIR      = os.path.abspath(".")
FIGURES_DIR = os.path.join(NB_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

def save_fig(name, source_prefix, tight=True):
    dest = os.path.join(FIGURES_DIR, source_prefix)
    os.makedirs(dest, exist_ok=True)
    path = os.path.join(dest, f"{name}.png")
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    print(f"  Saved -> figures/{source_prefix}/{name}.png")

print("Setup OK — figures ->", FIGURES_DIR)


In [ ]:

def draw_schema(title, columns, source_prefix, filename="schema"):
    n_rows = len(columns)
    fig_h  = max(3.0, 0.48 * n_rows + 1.6)
    fig, ax = plt.subplots(figsize=(13, fig_h))
    ax.axis("off")
    fig.suptitle(title, fontsize=14, fontweight="bold", y=0.99, ha="center")
    headers = ["Colonne", "Type", "Description"]
    col_x   = [0.0, 0.22, 0.37]
    col_w   = [0.22, 0.15, 0.63]
    row_h   = 1.0 / (n_rows + 1)
    for i, (hdr, x, w) in enumerate(zip(headers, col_x, col_w)):
        rect = mpatches.FancyBboxPatch(
            (x, 1 - row_h), w - 0.006, row_h * 0.88,
            boxstyle="round,pad=0.01", linewidth=0.5,
            edgecolor="#90A4AE", facecolor="#1565C0",
            transform=ax.transAxes, clip_on=False)
        ax.add_patch(rect)
        ax.text(x + w/2, 1 - row_h/2, hdr,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=11, fontweight="bold", color="white")
    type_colors = {
        "str":"#1565C0","int":"#2E7D32","float":"#6A1B9A",
        "datetime":"#E65100","bool":"#AD1457","date":"#00695C",
    }
    for ridx, (col_name, col_type, col_desc) in enumerate(columns):
        y_top = 1 - (ridx + 2) * row_h
        bg = "#F5F7FA" if ridx % 2 == 0 else "white"
        for x, w in zip(col_x, col_w):
            rect = mpatches.FancyBboxPatch(
                (x, y_top), w - 0.006, row_h * 0.88,
                boxstyle="round,pad=0.01", linewidth=0.5,
                edgecolor="#CFD8DC", facecolor=bg,
                transform=ax.transAxes, clip_on=False)
            ax.add_patch(rect)
        y_mid = y_top + row_h * 0.44
        tc = type_colors.get(col_type.split("[")[0].lower(), "#37474F")
        ax.text(col_x[0]+col_w[0]/2, y_mid, col_name,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=10, fontweight="bold", color="#1A237E")
        ax.text(col_x[1]+col_w[1]/2, y_mid, col_type,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=9, fontfamily="monospace", color=tc)
        ax.text(col_x[2]+0.01, y_mid, col_desc,
                transform=ax.transAxes, ha="left", va="center",
                fontsize=9.5, color="#37474F")
    plt.subplots_adjust(top=0.92, bottom=0)
    save_fig(filename, source_prefix)
    plt.show()


In [ ]:
PREFIX = "05_air_quality"
draw_schema(
    "Bronze Schema — Qualite de l'Air Airparif (ATMO)",
    [
        ("commune_code",    "str",      "Code INSEE commune parisienne (751xx)"),
        ("arrondissement",  "int",      "Numero arrondissement (1-20)"),
        ("date_prevision",  "str",      "Date de prevision (YYYY-MM-DD)"),
        ("indice_atmo",     "str",      "Indice global : Bon | Moyen | Degrade | Mauvais | Tres mauvais | Extremement mauvais"),
        ("indice_atmo_num", "int",      "Indice numerique (1=Bon -> 6=Extremement mauvais)"),
        ("no2",             "str",      "Indice qualite dioxyde d'azote"),
        ("o3",              "str",      "Indice qualite ozone"),
        ("pm10",            "str",      "Indice qualite particules PM10"),
        ("pm25",            "str",      "Indice qualite particules PM2.5"),
        ("so2",             "str",      "Indice qualite dioxyde de soufre"),
        ("ingested_at",     "datetime", "Horodatage UTC d'ingestion"),
    ], PREFIX
)

In [ ]:
aq_files = sorted(glob.glob(f"{BRONZE_ROOT}/air_quality/**/*.parquet", recursive=True))
if aq_files:
    df = pd.concat([pd.read_parquet(f) for f in aq_files], ignore_index=True)
else:
    rng = np.random.default_rng(42)
    indices = ["Bon","Moyen","Degrade","Mauvais","Tres mauvais"]
    rows = []
    for arr in range(1,21):
        for d in pd.date_range("2025-01-01","2026-06-01",freq="D"):
            idx = max(1,min(6,int(rng.normal(2.2,0.8))))
            rows.append({"commune_code":f"751{arr:02d}","arrondissement":arr,
                "date_prevision":d.strftime("%Y-%m-%d"),"indice_atmo":indices[min(idx-1,4)],
                "indice_atmo_num":idx,"no2":rng.choice(indices[:3]),"o3":rng.choice(indices[:4]),
                "pm10":rng.choice(indices[:3]),"pm25":rng.choice(indices[:3]),"so2":"Bon"})
    df = pd.DataFrame(rows)
df["date_prevision"] = pd.to_datetime(df["date_prevision"])
print(f"Shape : {df.shape}")
df.head(3)

In [ ]:
ATMO_COLORS = {"Bon":"#4CAF50","Moyen":"#FFEB3B","Degrade":"#FF9800",
    "Mauvais":"#F44336","Tres mauvais":"#9C27B0","Extremement mauvais":"#4A148C"}
fig, axes = plt.subplots(1,2,figsize=(15,5))
counts = df["indice_atmo"].value_counts()
axes[0].bar(counts.index, counts.values, color=[ATMO_COLORS.get(k,"#999") for k in counts.index], edgecolor="white",width=0.7)
axes[0].set_title("Distribution des indices ATMO globaux"); axes[0].set_xlabel("Indice ATMO"); axes[0].set_ylabel("Observations")
plt.setp(axes[0].xaxis.get_majorticklabels(),rotation=30,ha="right")
axes[1].hist(df["indice_atmo_num"].dropna(),bins=range(1,8),align="left",color=PALETTE["primary"],edgecolor="white",rwidth=0.8)
axes[1].set_xticks(range(1,7)); axes[1].set_xticklabels(["1 Bon","2 Moyen","3 Degrade","4 Mauvais","5 Tres
mauvais","6 Extrmt"])
axes[1].set_title("Distribution indice numerique ATMO"); axes[1].set_ylabel("Observations")
save_fig("distribution_atmo", PREFIX); plt.show()

In [ ]:
pols = ["no2","o3","pm10","pm25","so2"]; labs = ["NO2","O3","PM10","PM2.5","SO2"]
ATMO_COLORS = {"Bon":"#4CAF50","Moyen":"#FFEB3B","Degrade":"#FF9800",
    "Mauvais":"#F44336","Tres mauvais":"#9C27B0","Extremement mauvais":"#4A148C"}
fig, axes = plt.subplots(1,len(pols),figsize=(18,5))
for ax, pol, lab in zip(axes, pols, labs):
    counts = df[pol].value_counts()
    ax.bar(counts.index, counts.values, color=[ATMO_COLORS.get(k,"#999") for k in counts.index], edgecolor="white")
    ax.set_title(lab,fontsize=13)
    plt.setp(ax.xaxis.get_majorticklabels(),rotation=45,ha="right")
    if ax is axes[0]: ax.set_ylabel("Observations")
fig.suptitle("Qualite par polluant — Paris",fontsize=14,fontweight="bold")
save_fig("qualite_par_polluant", PREFIX); plt.show()

In [ ]:
daily = df.groupby("date_prevision")["indice_atmo_num"].mean().reset_index()
fig, ax = plt.subplots(figsize=(14,5))
ax.plot(daily["date_prevision"],daily["indice_atmo_num"],color=PALETTE["primary"],linewidth=1.2,alpha=0.8)
ax.fill_between(daily["date_prevision"],daily["indice_atmo_num"],1,alpha=0.12,color=PALETTE["primary"])
ax.set_yticks(range(1,7)); ax.set_yticklabels(["1 Bon","2 Moyen","3 Degrade","4 Mauvais","5 Tres mauvais","6 Extrmt"])
ax.set_title("Evolution journaliere de l'indice ATMO moyen — Paris"); ax.set_xlabel("Date")
save_fig("evolution_atmo", PREFIX); plt.show()